In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.models import VGG16_Weights

In [2]:
!pip install pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 54.5 MB/s  0:00:00


In [3]:
!pip install kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [kagglehub]/4 [kagglesdk]


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ultralytics/coco128")

print("Path to dataset files:", path)

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 6.66M/6.66M [00:00<00:00, 7.21MB/s]

Extracting files...
Path to dataset files: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3


In [9]:
import os

dataset_root = "/home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3"

for root, dirs, files in os.walk(dataset_root):
    print("DIR:", root)
    print("Subdirs:", dirs)
    print("Files:", files[:5])
    print("-" * 50)

DIR: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3
Subdirs: ['coco128']
Files: []
--------------------------------------------------
DIR: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128
Subdirs: ['labels', 'images']
Files: ['README.txt', 'LICENSE']
--------------------------------------------------
DIR: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/labels
Subdirs: ['train2017']
Files: []
--------------------------------------------------
DIR: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/labels/train2017
Subdirs: []
Files: ['000000000294.txt', '000000000036.txt', '000000000471.txt', '000000000491.txt', '000000000315.txt']
--------------------------------------------------
DIR: /home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/images
Subdirs: ['train2017']
Files: []
--------------------------------------------------
DIR: /home/onyxia/.cache/ka

In [10]:
import glob

img_dir = dataset_root + "/coco128/images/train2017"
lbl_dir = dataset_root + "/coco128/labels/train2017"

images = sorted(glob.glob(img_dir + "/*.jpg"))
labels = sorted(glob.glob(lbl_dir + "/*.txt"))

print("Images:", len(images))
print("Labels:", len(labels))
print(images[0])
print(labels[0])

Images: 128
Labels: 128
/home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/images/train2017/000000000009.jpg
/home/onyxia/.cache/kagglehub/datasets/ultralytics/coco128/versions/3/coco128/labels/train2017/000000000009.txt


In [11]:
len(images)

128

In [12]:
from dataloader import DataSSD300,DataSplitter

In [13]:
coco128loader=DataSSD300(img_dir,lbl_dir,gt_normalised=True)
splitter=DataSplitter(20,0.15,0.15)
train_dataloader, val_dataloader, test_dataloader=splitter(coco128loader)

In [34]:
for image, labels_list, gt_box_list in train_dataloader:
    print(image.shape)
 
    print((gt_box_list[0].shape))
    print((labels_list[0].shape))
    print((labels_list[0].dtype))
    break

torch.Size([20, 3, 300, 300])
torch.Size([11, 4])
torch.Size([11])
torch.int64


In [14]:
vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
vgg[16] = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)
base=nn.ModuleList(vgg[:30])#until 5_3 layer
base




1.3%

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /home/onyxia/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100.0%


ModuleList(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
  (17): Conv2d(256, 512, kernel_siz

In [20]:
from utils import setup_device_and_seed

device = setup_device_and_seed(seed=42)

In [35]:
from ssd import SSD

model=SSD(base,nb_classes=21,phase="test",alpha=1,device=device,prob_thr=0.01,nms_thr=0.45,top_k=200,variances=(0.1,0.2))
model=model.to(device)

In [36]:
model.phase="train"

In [40]:
c,l=model(x)

In [22]:
x=torch.randn(2,3,300,300).to(device)


In [23]:
f=model(x)


In [25]:
f=f[:,:4,:]

In [26]:
f[0]

tensor([[ 0.7852,  0.5991,  0.9499,  0.8460,  0.8345, 17.0000],
        [ 0.5131,  0.2206,  0.6243,  0.4284,  0.7952, 17.0000],
        [ 0.5535,  0.2052,  0.6765,  0.4358,  0.7685, 17.0000],
        [ 0.1503,  0.1811,  0.2822,  0.4103,  0.7521, 17.0000]],
       grad_fn=<SelectBackward0>)

In [28]:
torch.tensor([[0.1, 0.2, 0.5, 0.6], [0.3, 0.4, 0.7, 0.8]], dtype=torch.float32).shape

torch.Size([2, 4])

In [30]:
torch.tensor([0, 5], dtype=torch.int32).shape

torch.Size([2])

In [22]:
preds=[]

from torchmetrics.detection import MeanAveragePrecision

for i in range(f.shape[0]):

    actual_preds=f[i][~(f[i] == 0).all(dim=1)]
    preds.append({
        "boxes" : actual_preds[:,:4],
        "scores" : actual_preds[:,4],
        "labels" : actual_preds[:,5].to(torch.int32)-1
    })

targets = [
    {
        "boxes": torch.tensor([[0.1, 0.2, 0.5, 0.6], [0.3, 0.4, 0.7, 0.8]], dtype=torch.float32),
        "labels": torch.tensor([0, 5], dtype=torch.int32),
    },
    {
        "boxes": torch.tensor([[0.2, 0.3, 0.6, 0.9]], dtype=torch.float32),
        "labels": torch.tensor([3], dtype=torch.int32),
    },
]
metric = MeanAveragePrecision()
metric.update(preds, targets)
from pprint import pprint
pprint(metric.compute())


{'classes': tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19], dtype=torch.int32),
 'map': tensor(0.),
 'map_50': tensor(0.),
 'map_75': tensor(0.),
 'map_large': tensor(-1.),
 'map_medium': tensor(-1.),
 'map_per_class': tensor(-1.),
 'map_small': tensor(0.),
 'mar_1': tensor(0.),
 'mar_10': tensor(0.),
 'mar_100': tensor(0.),
 'mar_100_per_class': tensor(-1.),
 'mar_large': tensor(-1.),
 'mar_medium': tensor(-1.),
 'mar_small': tensor(0.)}


c:\Users\arthu\FastCrowdVision\my_env\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


In [38]:
!pip install faster-coco-eval

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for faster-coco-eval: filename=faster_coco_eval-1.7.2-cp314-cp314-win_amd64.whl size=331179 sha256=cefcc77999631f06d755aed75eb2cf4a1fdf692b0d2c467431ca3536899c83dc
  Stored in directory: c:\users\arthu\appdata\local\pip\cache\wheels\24\ba\f3\2b8753f13b8a76a257a95b1df1b87396e849c246acdb7d8b10
Successfully built faster-coco-eval



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
!pip install torchmetrics

   ---------------------------------------- 0.0/983.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/983.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/983.2 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/983.2 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/983.2 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/983.2 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/983.2 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/983.2 kB ? eta -:--:--
   -------------------- ----------------- 524.3/983.2 kB 266.9 kB/s eta 0:00:02
   -------------------- ----------------- 524.3/983.2 kB 266.9 kB/s eta 0:00:02
   -------------------- ----------------- 524.3/983.2 kB 266.9 kB/s eta 0:00:02
   ------------------------------ ------- 786.4/983.2 kB 293.6 kB/s eta 0:00:01
   ------------------------------ ------- 786.4/983.2 kB 293.6 kB/s eta 0:00:01
   -------------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:

from torchmetrics.detection import MeanAveragePrecision





In [11]:
from train import train



In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, weight_decay = 0.0005, momentum = 0.9)
train(model,optimizer,train_dataloader,val_dataloader,modelname="ssd_coco128V1",device=device)